# 04 - Ewaluacja: RAFT vs RAG Baseline

Ten notebook:
1. Buduje RAG baseline (ChromaDB + vanilla Mistral 7B)
2. Generuje odpowiedzi obu modeli na test set
3. Porównuje wyniki za pomocą metryk automatycznych + LLM-as-a-Judge
4. Wizualizuje wyniki

## 1. Setup

In [ ]:
%%capture
!pip install chromadb sentence-transformers langchain langchain-community
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install google-generativeai matplotlib seaborn

In [ ]:
import json
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Dodaj src/ do path
sys.path.insert(0, "../src" if Path("../src").exists() else "src")

from evaluation import (
    compute_conspiracy_rejection_rate,
    judge_comparison,
    run_evaluation_suite,
    save_results,
    print_comparison_table,
)

# Konfiguracja
os.environ["GEMINI_API_KEY"] = " "
TEST_DATA_PATH = "data/raft_test.jsonl"
RAFT_MODEL_PATH = "outputs/raft-mythbuster-mistral-7b-lora"

## 2. Przygotowanie test set

In [ ]:
# Ładowanie danych testowych
def load_jsonl(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

test_data = load_jsonl(TEST_DATA_PATH)
print(f"Test set: {len(test_data)} pytań")

test_questions = [d["question"] for d in test_data]
test_contexts = [d["context"] for d in test_data]
test_golden_docs = [d.get("golden_doc_content", "") for d in test_data]
test_reference_answers = [d["answer"] for d in test_data]

## 3. RAG Baseline: ChromaDB + Vanilla Mistral 7B

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Embedding model (wspiera PL)
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# ChromaDB setup
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="raft_docs",
    metadata={"hnsw:space": "cosine"},
)

golden_docs = load_jsonl("data/golden_docs.jsonl")
distractors = load_jsonl("data/distractors.jsonl")
all_docs = golden_docs + distractors

for i, doc in enumerate(all_docs):
    collection.add(
        documents=[doc["content"]],
        ids=[f"doc_{i}"],
        metadatas=[{"type": doc["type"], "source": doc.get("source", "unknown")}],
    )

print(f"Zaindeksowano {len(all_docs)} dokumentów w ChromaDB")
print(f"  - Golden docs: {len(golden_docs)}")
print(f"  - Distractors: {len(distractors)}")

In [ ]:
def rag_retrieve(question: str, n_results: int = 5) -> str:
    """Retrieval z ChromaDB — zwraca top-k dokumentów jako kontekst."""
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
    )
    
    context_parts = []
    for i, doc in enumerate(results["documents"][0]):
        label = chr(ord("A") + i)
        context_parts.append(f"[Dokument {label}]\n{doc}")
    
    return "\n\n---\n\n".join(context_parts)

## 4. Ładowanie modeli

In [ ]:
from unsloth import FastLanguageModel
import torch

# --- Model RAFT (fine-tuned) ---
raft_model, raft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=RAFT_MODEL_PATH,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(raft_model)
print("Model RAFT załadowany.")

# --- Model RAG Baseline (vanilla, bez fine-tuningu) ---
baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(baseline_model)
print("Model Baseline załadowany.")

In [ ]:
INFERENCE_PROMPT = """### Instruction:
Jesteś ekspertem od weryfikacji faktów. Na podstawie podanego kontekstu odpowiedz na pytanie.
Cytuj wiarygodne źródła dosłownie (używając ##begin_quote## i ##end_quote##).
Zidentyfikuj i odrzuć dezinformację, wskazując błędy logiczne.

### Context:
{context}

### Question:
{question}

### Response:
"""

def generate_response(model, tokenizer, question: str, context: str, max_tokens: int = 512) -> str:
    """Generuje odpowiedź modelu na pytanie z kontekstem."""
    prompt = INFERENCE_PROMPT.format(context=context, question=question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return response.strip()

## 5. Generowanie predykcji

In [ ]:
from tqdm import tqdm

raft_predictions = []
rag_predictions = []

for i, (question, context) in enumerate(tqdm(
    zip(test_questions, test_contexts), total=len(test_questions), desc="Generowanie"
)):
    # RAFT: używa kontekstu z test set (jak w treningu)
    raft_pred = generate_response(raft_model, raft_tokenizer, question, context)
    raft_predictions.append(raft_pred)
    
    # RAG Baseline: robi własny retrieval z ChromaDB
    rag_context = rag_retrieve(question, n_results=5)
    rag_pred = generate_response(baseline_model, baseline_tokenizer, question, rag_context)
    rag_predictions.append(rag_pred)

print(f"\nWygenerowano {len(raft_predictions)} predykcji RAFT")
print(f"Wygenerowano {len(rag_predictions)} predykcji RAG Baseline")

## 6. Ewaluacja automatyczna

In [ ]:
# Metryki custom 
results = run_evaluation_suite(
    test_questions=test_questions,
    test_contexts=test_contexts,
    test_golden_docs=test_golden_docs,
    raft_predictions=raft_predictions,
    rag_predictions=rag_predictions,
    judge_model=None,
)

print_comparison_table(results)

## 7. LLM-as-a-Judge (Gemini Pro)

In [ ]:
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
judge = genai.GenerativeModel("gemini-2.5-pro")

# Re-run z judge
results_with_judge = run_evaluation_suite(
    test_questions=test_questions,
    test_contexts=test_contexts,
    test_golden_docs=test_golden_docs,
    raft_predictions=raft_predictions,
    rag_predictions=rag_predictions,
    judge_model=judge,
)

print_comparison_table(results_with_judge)

# Zapis
save_results(results_with_judge, "outputs/evaluation_results.json")

## 8. Wizualizacja wyników

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Wykres 1: CRR porównanie ---
crr_metrics = ["conspiracy_rejection_rate", "disinfo_identification_rate"]
crr_labels = ["Conspiracy\nRejection", "Disinfo\nIdentification"]

raft_crr_vals = [results["raft_metrics"]["crr"].get(m, 0) for m in crr_metrics]
rag_crr_vals = [results["rag_metrics"]["crr"].get(m, 0) for m in crr_metrics]

x = np.arange(len(crr_labels))
width = 0.35

axes[0].bar(x - width/2, raft_crr_vals, width, label="RAFT", color="#2ecc71")
axes[0].bar(x + width/2, rag_crr_vals, width, label="RAG Baseline", color="#e74c3c")
axes[0].set_xlabel("Metryka")
axes[0].set_ylabel("Wartość")
axes[0].set_title("Conspiracy Rejection Metrics")
axes[0].set_xticks(x)
axes[0].set_xticklabels(crr_labels)
axes[0].legend()
axes[0].set_ylim(0, 1)

# --- Wykres 2: LLM Judge wins ---
judge_data = results_with_judge.get("comparison", {}).get("llm_judge", {})
if judge_data:
    wins = [judge_data.get("raft_wins", 0), judge_data.get("rag_wins", 0), judge_data.get("ties", 0)]
    labels_pie = ["RAFT wins", "RAG wins", "Ties"]
    colors_pie = ["#2ecc71", "#e74c3c", "#95a5a6"]
    axes[1].pie(wins, labels=labels_pie, colors=colors_pie, autopct="%1.0f%%", startangle=90)
    axes[1].set_title("LLM-as-a-Judge: Win Rate")
else:
    axes[1].text(0.5, 0.5, "No Judge Data", ha="center", va="center")
    axes[1].set_title("LLM-as-a-Judge")

# --- Wykres 3: Średnie scores (jeśli dostępne) ---
avg_raft = results_with_judge.get("comparison", {}).get("avg_raft_scores", {})
avg_rag = results_with_judge.get("comparison", {}).get("avg_rag_scores", {})

if avg_raft and avg_rag:
    categories = list(avg_raft.keys())
    raft_vals = [avg_raft[c] for c in categories]
    rag_vals = [avg_rag[c] for c in categories]
    
    x = np.arange(len(categories))
    axes[2].bar(x - width/2, raft_vals, width, label="RAFT", color="#2ecc71")
    axes[2].bar(x + width/2, rag_vals, width, label="RAG Baseline", color="#e74c3c")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(categories, rotation=45, ha="right")
    axes[2].set_ylabel("Score (1-5)")
    axes[2].set_title("Średnie oceny sędziego")
    axes[2].legend()
    axes[2].set_ylim(0, 5)
else:
    axes[2].text(0.5, 0.5, "No Score Data", ha="center", va="center")

plt.tight_layout()
plt.savefig("outputs/evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Wykresy zapisane do: outputs/evaluation_plots.png")

Podsumowanie

In [ ]:
print("\n" + "="*70)
print("  PODSUMOWANIE EKSPERYMENTU")
print("="*70)
print(f"\nModel bazowy: Mistral 7B v0.3 (4-bit)")
print(f"Metoda: RAFT (Retrieval-Augmented Fine-Tuning)")
print(f"Dataset treningowy: ~200 przykładów (P=80%)")
print(f"Test set: {len(test_questions)} pytań")
print(f"Tematyka: COVID-19 / szczepionki (polski)")
print(f"\nKluczowe wnioski:")

raft_crr = results["raft_metrics"]["crr"]["conspiracy_rejection_rate"]
rag_crr = results["rag_metrics"]["crr"]["conspiracy_rejection_rate"]

if raft_crr > rag_crr:
    print(f" RAFT wykazuje WYŻSZĄ odporność na dezinformację ({raft_crr:.1%} vs {rag_crr:.1%})")
else:
    print(f" Brak wyraźnej poprawy CRR (RAFT: {raft_crr:.1%}, RAG: {rag_crr:.1%})")

raft_qu = results["raft_metrics"]["quote_usage_rate"]
rag_qu = results["rag_metrics"]["quote_usage_rate"]
if raft_qu > rag_qu:
    print(f" RAFT częściej cytuje źródła ({raft_qu:.1%} vs {rag_qu:.1%})")

print(f"\n{'='*70}")